# Task 3: Future Forecasting
## Generating 6-12 Month Forecasts Using Best Model

This notebook generates future forecasts using the best-performing model from Task 2.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import MinMaxScaler

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## 1. Load Data and Model Comparison Results

In [ ]:
# Load data
prices = pd.read_csv('../data/raw/stock_prices.csv', index_col=0, parse_dates=True)
model_comparison = pd.read_csv('../data/processed/model_comparison.csv')

print("Data loaded successfully!")
print(f"Price data shape: {prices.shape}")
print(f"Date range: {prices.index.min()} to {prices.index.max()}")

print("\n=== Model Comparison from Task 2 ===")
display(model_comparison)

## 2. Select Best Model for Each Asset

Based on MAPE (Mean Absolute Percentage Error), we select the model with lowest error.

In [ ]:
# Identify best model based on MAPE
best_model_name = model_comparison.loc[model_comparison['MAPE (%)'].idxmin(), 'Model']
best_mape = model_comparison['MAPE (%)'].min()

print(f"=" * 60)
print("BEST MODEL SELECTION")
print(f"=" * 60)
print(f"\nBest performing model: {best_model_name}")
print(f"MAPE: {best_mape:.2f}%")
print(f"\nThis model will be used for future forecasting.")

## 3. Generate 6-Month and 12-Month Forecasts for TSLA

Using ARIMA as the primary forecasting model (most reliable for financial data).

In [ ]:
# Focus on TSLA for portfolio forecasting
TICKER = 'TSLA'
target_data = prices[TICKER].dropna()

print(f"=" * 60)
print(f"FORECASTING FOR {TICKER}")
print(f"=" * 60)
print(f"\nHistorical data: {len(target_data)} observations")
print(f"Date range: {target_data.index.min()} to {target_data.index.max()}")

In [ ]:
# Define forecast horizons
FORECAST_6M = 126   # ~6 months of trading days
FORECAST_12M = 252  # ~12 months of trading days

# Fit ARIMA model on full historical data
# Using (1, 1, 1) as baseline - can be optimized
ARIMA_ORDER = (2, 1, 2)

print(f"\n### Fitting ARIMA{ARIMA_ORDER} on full historical data ###")

arima_full = ARIMA(target_data, order=ARIMA_ORDER)
arima_fitted = arima_full.fit()

print(arima_fitted.summary())

In [ ]:
# Generate 6-month forecast
print(f"\n### 6-Month Forecast ({FORECAST_6M} trading days) ###")

forecast_6m_result = arima_fitted.get_forecast(steps=FORECAST_6M)
forecast_6m = forecast_6m_result.predicted_mean
conf_6m = forecast_6m_result.conf_int(alpha=0.05)

# Create future dates
last_date = target_data.index[-1]
future_dates_6m = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=FORECAST_6M, freq='B')

forecast_6m.index = future_dates_6m
conf_6m.index = future_dates_6m

print(f"Forecast period: {future_dates_6m[0].date()} to {future_dates_6m[-1].date()}")
print(f"\nForecast Statistics:")
print(f"  Starting price: ${forecast_6m.iloc[0]:.2f}")
print(f"  Ending price: ${forecast_6m.iloc[-1]:.2f}")
print(f"  Min forecast: ${forecast_6m.min():.2f}")
print(f"  Max forecast: ${forecast_6m.max():.2f}")
print(f"  Mean forecast: ${forecast_6m.mean():.2f}")

In [ ]:
# Generate 12-month forecast
print(f"\n### 12-Month Forecast ({FORECAST_12M} trading days) ###")

forecast_12m_result = arima_fitted.get_forecast(steps=FORECAST_12M)
forecast_12m = forecast_12m_result.predicted_mean
conf_12m = forecast_12m_result.conf_int(alpha=0.05)

# Create future dates
future_dates_12m = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=FORECAST_12M, freq='B')

forecast_12m.index = future_dates_12m
conf_12m.index = future_dates_12m

print(f"Forecast period: {future_dates_12m[0].date()} to {future_dates_12m[-1].date()}")
print(f"\nForecast Statistics:")
print(f"  Starting price: ${forecast_12m.iloc[0]:.2f}")
print(f"  Ending price: ${forecast_12m.iloc[-1]:.2f}")
print(f"  Min forecast: ${forecast_12m.min():.2f}")
print(f"  Max forecast: ${forecast_12m.max():.2f}")
print(f"  Mean forecast: ${forecast_12m.mean():.2f}")

## 4. Visualization: Historical Data + Test Predictions + Future Forecast

In [ ]:
# Load test predictions from Task 2 (if available)
try:
    test_predictions = pd.read_csv('../data/processed/test_predictions.csv', index_col=0, parse_dates=True)
    has_test_preds = True
except:
    has_test_preds = False
    print("Note: Test predictions not found. Showing historical + forecast only.")

In [ ]:
# Visualization: Historical vs Future Forecast
fig, ax = plt.subplots(figsize=(16, 8))

# Historical data (last 2 years for context)
historical_period = target_data[-504:]  # Last 2 years
historical_period.plot(ax=ax, label='Historical Prices', color='blue', linewidth=1.5)

# 6-month forecast
forecast_6m.plot(ax=ax, label='6-Month Forecast', color='green', linewidth=2)
ax.fill_between(conf_6m.index, conf_6m.iloc[:, 0], conf_6m.iloc[:, 1], 
                 color='green', alpha=0.2, label='6-Month 95% CI')

# 12-month forecast (extension)
forecast_12m[FORECAST_6M:].plot(ax=ax, label='12-Month Forecast (extended)', 
                                 color='orange', linewidth=2, linestyle='--')
ax.fill_between(conf_12m.index[FORECAST_6M:], conf_6m.iloc[-1, 0], conf_12m.iloc[FORECAST_6M:, 1], 
                 color='orange', alpha=0.1, label='12-Month 95% CI (extended)')

# Mark transition points
ax.axvline(x=target_data.index[-1], color='red', linestyle=':', linewidth=2, label='Forecast Start')
ax.axvline(x=future_dates_6m[-1], color='green', linestyle='--', alpha=0.5, label='6-Month Mark')

ax.set_title(f'{TICKER} Future Price Forecast (6-Month and 12-Month)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/future_forecast.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/future_forecast.png")

In [ ]:
# Detailed 12-Month Forecast Visualization
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Top: Full forecast with confidence intervals
ax1 = axes[0]
target_data[-252:].plot(ax=ax1, label='Historical (Last Year)', color='blue', linewidth=2)
forecast_12m.plot(ax=ax1, label='12-Month Forecast', color='red', linewidth=2)
ax1.fill_between(conf_12m.index, conf_12m.iloc[:, 0], conf_12m.iloc[:, 1], 
                  color='red', alpha=0.2, label='95% Confidence Interval')

ax1.axvline(x=target_data.index[-1], color='black', linestyle='--', linewidth=2, label='Forecast Start')
ax1.set_title(f'{TICKER} 12-Month Forecast with 95% Confidence Intervals', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bottom: Forecast uncertainty over time
ax2 = axes[1]
uncertainty = (conf_12m.iloc[:, 1] - conf_12m.iloc[:, 0]) / 2
uncertainty.plot(ax=ax2, color='purple', linewidth=2)
ax2.fill_between(uncertainty.index, 0, uncertainty, alpha=0.3, color='purple')
ax2.set_title('Forecast Uncertainty (Half-width of 95% CI)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Price Range ($)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/forecast_uncertainty.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/forecast_uncertainty.png")

## 5. Expected Return Calculation for Portfolio Optimization

Calculate the expected annualized return for TSLA based on the forecast.

In [ ]:
# Calculate expected return from forecast
current_price = target_data.iloc[-1]
forecast_price_6m = forecast_6m.iloc[-1]
forecast_price_12m = forecast_12m.iloc[-1]

# Expected returns
expected_return_6m = (forecast_price_6m - current_price) / current_price
expected_return_12m = (forecast_price_12m - current_price) / current_price

# Annualize 6-month return
annualized_return_6m = (1 + expected_return_6m) ** 2 - 1

print(f"=" * 60)
print(f"EXPECTED RETURNS FOR {TICKER}")
print(f"=" * 60)
print(f"\nCurrent Price: ${current_price:.2f}")
print(f"\n6-Month Forecast:")
print(f"  Ending Price: ${forecast_price_6m:.2f}")
print(f"  6-Month Return: {expected_return_6m:.2%}")
print(f"  Annualized Return: {annualized_return_6m:.2%}")
print(f"\n12-Month Forecast:")
print(f"  Ending Price: ${forecast_price_12m:.2f}")
print(f"  12-Month Return: {expected_return_12m:.2%}")
print(f"\nConfidence Interval (12-Month):")
print(f"  Lower Bound: ${conf_12m.iloc[-1, 0]:.2f}")
print(f"  Upper Bound: ${conf_12m.iloc[-1, 1]:.2f}")
print(f"  95% CI Return Range: {((conf_12m.iloc[-1, 0] - current_price) / current_price):.2%} to {((conf_12m.iloc[-1, 1] - current_price) / current_price):.2%}")

In [ ]:
# Save forecast results for Task 4
forecast_results = pd.DataFrame({
    'Ticker': [TICKER],
    'Current_Price': [current_price],
    'Forecast_6M': [forecast_price_6m],
    'Forecast_12M': [forecast_price_12m],
    'Annualized_Return': [expected_return_12m],
    'CI_Lower': [conf_12m.iloc[-1, 0]],
    'CI_Upper': [conf_12m.iloc[-1, 1]]
})

forecast_results.to_csv('../data/processed/tsla_forecast.csv', index=False)

# Also save the full forecast series
forecast_12m.to_csv('../data/processed/tsla_forecast_series.csv')

print("\nForecast results saved for portfolio optimization.")
display(forecast_results)

## Summary

### Key Outputs:
1. **Best Model Selected**: ARIMA based on MAPE performance
2. **6-Month Forecast**: Price projection with confidence intervals
3. **12-Month Forecast**: Extended price projection
4. **Expected Return**: Annualized return for portfolio optimization

### Next Steps:
- Use TSLA forecast in Task 4 portfolio optimization
- Combine with BND and SPY historical returns
- Generate efficient frontier and optimal portfolios